# 07 — Network Optimization (Sequential Greedy)

**Sequential greedy station placement.**  
Identifies AFIR coverage gaps on interurban roads. Places new stations at high-demand gap
midpoints (or nearest service area) using sequential scoring: `V_i = n_chargers_needed × gap_length_km`.

Inputs: interurban roads + existing charger baseline + ABM demand per segment  
Output: `proposed_stations.csv`

## Data Inputs
- `data/processed/interurban_roads.parquet` — road network with geometry
- `data/processed/interurban_chargers_baseline.csv` — existing chargers
- `data/processed/demand_per_segment.csv` — ABM demand from NB06
- `data/processed/service_areas_clean.geojson` — preferred candidate sites

## Data Output
- `data/processed/proposed_stations.csv`
  - Columns: `location_id, latitude, longitude, route_segment, n_chargers_proposed`

In [ ]:
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

sys.path.append('..')
from src.constants import (
    MAX_STATION_SPACING_KM,
    MAX_STATION_SPACING_TENT_CORE_KM,
    MAX_STATION_SPACING_TENT_COMP_KM,
    MIN_EXISTING_CHARGER_POWER_KW,
    MIN_CHARGERS_STANDARD, MIN_CHARGERS_TENT,
)
from src.optimization import compute_coverage_gaps, place_stations_greedy

DATA_DIR = Path('../data/processed')
print('✅ Imports OK')
print(f'   AFIR spacing — TEN-T Core:   {MAX_STATION_SPACING_TENT_CORE_KM} km')
print(f'   AFIR spacing — TEN-T Comp:   {MAX_STATION_SPACING_TENT_COMP_KM} km')
print(f'   AFIR spacing — General:      {MAX_STATION_SPACING_KM} km')
print(f'   Min existing power for coverage: {MIN_EXISTING_CHARGER_POWER_KW} kW')

## Step 1: Load inputs

In [ ]:
# Road network
roads = gpd.read_parquet(DATA_DIR / 'interurban_roads.parquet')
print(f'📊 Roads: {len(roads):,} segments')

# Existing chargers baseline
chargers = pd.read_csv(DATA_DIR / 'interurban_chargers_baseline.csv')
print(f'🔌 Existing chargers (all): {len(chargers):,}')
fast_chargers = chargers[chargers['max_power_kw'] >= MIN_EXISTING_CHARGER_POWER_KW]
print(f'   Fast chargers (≥{MIN_EXISTING_CHARGER_POWER_KW} kW, valid coverage): {len(fast_chargers):,}')

# ABM demand output
demand = pd.read_csv(DATA_DIR / 'demand_per_segment.csv')
print(f'📈 Demand segments: {len(demand):,}')
print(f'   n_chargers range: {demand["n_chargers_needed"].min()} – {demand["n_chargers_needed"].max()}')

# Service areas (preferred candidate locations)
sa_path = DATA_DIR / 'service_areas_clean.geojson'
service_areas = gpd.read_file(sa_path) if sa_path.exists() else None
sa_count = len(service_areas) if service_areas is not None else 0
print(f'🅿️  Service areas available: {sa_count}')

## Step 2: Identify AFIR Coverage Gaps

For each road segment, find nearest fast charger (≥50 kW).
Flag segments where distance > AFIR threshold (60 / 100 / 120 km by road type).

In [ ]:
gaps = compute_coverage_gaps(
    road_segments_df=roads,
    existing_stations_df=chargers,
)

print(f'🔍 Coverage gap analysis:')
print(f'   Total segments: {len(roads):,}')
print(f'   Gap segments:   {len(gaps):,} ({len(gaps)/len(roads)*100:.1f}%)')

if len(gaps) > 0 and 'gap_spacing_threshold_km' in gaps.columns:
    labels = {60: 'TEN-T Core (60 km)', 100: 'TEN-T Comp (100 km)', 120: 'General (120 km)'}
    for thresh, count in gaps['gap_spacing_threshold_km'].value_counts().sort_index().items():
        print(f'   {labels.get(thresh, f"{thresh} km")}: {count:,} gap segments')

## Step 3: Sequential Greedy Placement

Score: `V_i = n_chargers_needed × gap_length_km`  
Select top candidate, mark covered road segments, repeat until all gaps closed.  
Prefer service area locations when within 5 km of gap midpoint.

In [ ]:
proposed = place_stations_greedy(
    gap_segments_df=gaps,
    demand_df=demand,
    service_areas_gdf=service_areas,
)

print(f'🏗️  Sequential greedy placement complete:')
print(f'   Proposed new stations: {len(proposed):,}')

if len(proposed) > 0:
    print(f'\n   Charger distribution per station:')
    print(proposed['n_chargers_proposed'].value_counts().sort_index().to_string())
    total_chargers = proposed['n_chargers_proposed'].sum()
    print(f'\n   Total chargers: {total_chargers:,}  |  Total capacity: {total_chargers*150:,} kW')

## Step 4: Validation & Save

In [ ]:
if len(proposed) == 0:
    print('ℹ️  No gaps — creating empty proposed_stations.csv')
    proposed = pd.DataFrame(columns=[
        'location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed'
    ])
else:
    required = ['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed']
    missing = set(required) - set(proposed.columns)
    assert not missing, f'Missing columns: {missing}'
    assert proposed['latitude'].between(35.0, 44.5).all(), 'Latitude out of Spain bounds'
    assert proposed['longitude'].between(-10.0, 5.0).all(), 'Longitude out of Spain bounds'
    assert proposed['n_chargers_proposed'].between(2, 12).all(), 'Charger count out of range'
    assert proposed['location_id'].is_unique, 'Duplicate location_ids'
    print(f'✅ Validation passed: {len(proposed):,} stations, all checks OK')

out_path = DATA_DIR / 'proposed_stations.csv'
proposed[['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed']].to_csv(
    out_path, index=False
)
print(f'💾 Saved → {out_path}')
proposed.head()